# Day11. Tool calling

In [1]:
from dotenv import load_dotenv
import os, json


load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
openai_default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
gemini_key = os.getenv("GOOGLE_API_KEY")
gemini_default_model = os.getenv("GEMINI_DEFAULT_MODEL", "gemini-4-hydrogen")

In [2]:
##############################################
################ fallback 구조 ################
##############################################
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


g_llm = init_chat_model(f'google_genai:{gemini_default_model}')
o_llm = init_chat_model(f'openai:{openai_default_model}')

with_fallbacks_llm = o_llm.with_fallbacks([g_llm])

In [10]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

ret = RunnableParallel(
    add = (lambda arg: arg + 10),
    mult = (lambda arg: arg * 5),
)
result = ret.invoke(5)

print(result)

{'add': 15, 'mult': 25}


In [23]:
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

ret = RunnableParallel(
    add = RunnableLambda(lambda arg: arg + 10),
    mult = RunnableLambda(lambda arg: arg * 5)
)

prompt = ChatPromptTemplate.from_template(
    "더한 값은 {add}이고, 곱한 값은 {mult} 두 숫자의 차이를 설명해주세요."
)

chain = ret | prompt | o_llm

result = chain.invoke(5)

print(result)

content='두 수를 \\(x, y\\)라고 하면 \\(x+y=15\\), \\(xy=25\\)입니다. 두 수를 근으로 하는 방정식은\n\n\\[\nt^2-15t+25=0\n\\]\n\n이고, 두 수는 \\(\\frac{15+5\\sqrt5}{2}\\), \\(\\frac{15-5\\sqrt5}{2}\\)입니다. 따라서 두 수의 차이는\n\n\\[\n\\boxed{5\\sqrt5 \\approx 11.18}\n\\]\n\n입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 228, 'prompt_tokens': 29, 'total_tokens': 257, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 105, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Enw7mDh83eEKgOc8g5aaUf26rrpVV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'} id='lc_run--01a21053-1001-766c-b9e2-82daf04d9a24-0' tool_calls=[] invalid_tool_c

In [ ]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=openai_default_model)
parser = StrOutputParser()

# 장점을 설명하는 체인
pros_prompt = ChatPromptTemplate.from_template("{topic}의 장점을 한 문장으로 알려줘")
pros_chain = pros_prompt | llm | parser

# 단점을 설명하는 체인
cons_prompt = ChatPromptTemplate.from_template("{topic}의 단점을 한 문장으로 알려줘")
cons_chain = cons_prompt | llm | parser

# 두 체인을 병렬로 묶기
parallel_chain = RunnableParallel(pros=pros_chain, cons=cons_chain) 

result = parallel_chain.invoke({"topic": "전기차"})
print(result)
# 출력 예: {'pros': '전기차는 ...', 'cons': '전기차는 ...'}

In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

ret = RunnableParallel(
    original = RunnablePassthrough(),
    add = lambda x: x +10
)

prompt = ChatPromptTemplate.from_template('original의 값은: {original}이고 함수 실행 결과 값은 {add}')

chain = ret | prompt 
result = chain.invoke(5)
print(result)


messages=[HumanMessage(content='original의 값은: 5이고 함수 실행 결과 값은 15', additional_kwargs={}, response_metadata={})]


In [62]:
# 도구 코드

from langchain.tools import tool

@tool
def add(a:int, b:int)->int:
    '''두 수를 더한 값을 반환하는 함수'''
    return a+b

@tool
def get_word_length(word:str)->int:
    '''입력 받은 문자열의 수를 반환'''
    return len(word)

@tool
def minus(a:int, b:int)->int:
    '''두 수를 뺀 값을 반환하는 함수'''
    return a - b

@tool
def multi(a:int, b:int)->int:
    '''두 수를 곱한 값을 반환하는 함수'''
    return a*b


@tool
def devide(a:int, b:int)->int:
    '''두 수를 나눈 값을 반환하는 함수이고
    만약 b의 값이 0이면 None값 반환
    '''
    if b == 0:
        return None
    return a/b

tools = [add, get_word_length, minus, multi, devide]
tools_dict = {t.name: t for t in tools}
llm_with_tools = o_llm.bind_tools(tools)
print(tools_dict)

{'add': StructuredTool(name='add', description='두 수를 더한 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001575EFCC7C0>), 'get_word_length': StructuredTool(name='get_word_length', description='입력 받은 문자열의 수를 반환', args_schema=<class 'langchain_core.utils.pydantic.get_word_length'>, func=<function get_word_length at 0x000001575EFCDE40>), 'minus': StructuredTool(name='minus', description='두 수를 뺀 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.minus'>, func=<function minus at 0x000001575EFCF2E0>), 'multi': StructuredTool(name='multi', description='두 수를 곱한 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.multi'>, func=<function multi at 0x000001575EFCEF20>), 'devide': StructuredTool(name='devide', description='두 수를 나눈 값을 반환하는 함수이고\n만약 b의 값이 0이면 None값 반환', args_schema=<class 'langchain_core.utils.pydantic.devide'>, func=<function devide at 0x000001575EFCCD60>)}


In [28]:
query = '5와 7일 더해줘'

tools = [add, get_word_length]
tools_dict = {t.name: t for t in tools}
llm_with_tools = o_llm.bind_tools(tools)
result = llm_with_tools.invoke(query)

print(result)

content=[{'arguments': '{"a":5,"b":7}', 'call_id': 'call_6f98908db59d4ff79751b7fce41f437f', 'name': 'add', 'type': 'function_call', 'id': 'fc_01ea3fce05980a8d006b0644d0186c87d0bc5ba197260ce62a', 'status': 'completed'}] additional_kwargs={} response_metadata={'id': 'resp_01ea3fce05980a8d006b0644cf41c887d09dbb62225ff8c2fc', 'created_at': 1795572943.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'finish_reason': 'completed', 'normalized_finish_reason': 'tool_call'} id='resp_01ea3fce05980a8d006b0644cf41c887d09dbb62225ff8c2fc' tool_calls=[{'name': 'add', 'args': {'a': 5, 'b': 7}, 'id': 'call_6f98908db59d4ff79751b7fce41f437f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 21, 'total_tokens': 101, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


In [31]:
from langchain_core.messages import ToolMessage

tools = [add, get_word_length]
tools_dict = {t.name: t for t in tools}

llm_with_tools = o_llm.bind_tools(tools)

query = "5와 7을 더해 줘"

# 1차 호출: 모델이 도구 호출을 요청
res1 = llm_with_tools.invoke(query)
print("Tool 호출 정보:", res1.tool_calls)

# 도구 실행
tool_call = res1.tool_calls[0]
result = tools_dict[tool_call["name"]].invoke(tool_call["args"])
print("Tool 실행 결과:", result)

# 2차 호출: 도구 결과를 모델에 전달 → 최종 답변
messages = [
    ("human", query),
    res1,
    ToolMessage(tool_call_id=tool_call["id"], content=str(result)),
]
res2 = llm_with_tools.invoke(messages)
print("최종 답변:", res2.text)   # .text: 응답 텍스트만 문자열로 꺼냄 (7장 참고)

Tool 호출 정보: [{'name': 'add', 'args': {'a': 5, 'b': 7}, 'id': 'call_030b1116314249fc88dbc900511ce0ec', 'type': 'tool_call'}]
Tool 실행 결과: 12
최종 답변: 12


In [65]:
# 도구 코드

from langchain.tools import tool

@tool
def add(a:float, b:float)->float:
    '''두 수를 더한 값을 반환하는 함수'''
    return a+b

@tool
def get_word_length(word:str)->int:
    '''입력 받은 문자열의 수를 반환'''
    return len(word)

@tool
def minus(a:float, b:float)->float:
    '''두 수를 뺀 값을 반환하는 함수'''
    return a - b

@tool
def multi(a:float, b:float)->float:
    '''두 수를 곱한 값을 반환하는 함수'''
    return a*b


@tool
def devide(a:float, b:float)->float:
    '''두 수를 나눈 값을 반환하는 함수이고
    만약 b의 값이 0이면 None값 반환
    '''
    if b == 0:
        return None
    return a/b

tools = [add, get_word_length, minus, multi, devide]
tools_dict = {t.name: t for t in tools}
llm_with_tools = o_llm.bind_tools(tools)
print(tools_dict)

{'add': StructuredTool(name='add', description='두 수를 더한 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001575F000A40>), 'get_word_length': StructuredTool(name='get_word_length', description='입력 받은 문자열의 수를 반환', args_schema=<class 'langchain_core.utils.pydantic.get_word_length'>, func=<function get_word_length at 0x000001575F001940>), 'minus': StructuredTool(name='minus', description='두 수를 뺀 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.minus'>, func=<function minus at 0x000001575F000E00>), 'multi': StructuredTool(name='multi', description='두 수를 곱한 값을 반환하는 함수', args_schema=<class 'langchain_core.utils.pydantic.multi'>, func=<function multi at 0x000001575F001300>), 'devide': StructuredTool(name='devide', description='두 수를 나눈 값을 반환하는 함수이고\n만약 b의 값이 0이면 None값 반환', args_schema=<class 'langchain_core.utils.pydantic.devide'>, func=<function devide at 0x000001575F0007C0>)}


In [67]:
# query = "(((5+4)*3)%2)-10"
# query = "(((5+4)*3)/2)-10"
query = "5와 4를 더한 결과에 3을 곱하고, 그 결과를 2로 나눈 후 그 값에서 10을 빼줘"
messages = [("system", '툴이 있으면 내 툴을 먼저 사용해줘'),("human", query)]

for step in range(10):                     
    res = llm_with_tools.invoke(messages)
    print(res.tool_calls)
    messages.append(res)

    if not res.tool_calls:                 
        print("\n최종 답변:", res.text)
        break

    for call in res.tool_calls:
        tool_fn = tools_dict.get(call["name"])
        if tool_fn is None:
            result = f"오류: '{call['name']}' 도구가 없습니다."
        else:
            result = tool_fn.invoke(call["args"])
        print(f"[{step + 1}] {call['name']}({call['args']}) → {result}")
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)))
else:
    print("최대 반복 횟수에 도달했습니다. 최종 답변을 얻지 못했습니다.")

[{'name': 'add', 'args': {'a': 5, 'b': 4}, 'id': 'call_9b955e78ceb74181a9986aabe8475994', 'type': 'tool_call'}]
[1] add({'a': 5, 'b': 4}) → 9.0
[{'name': 'multi', 'args': {'a': 9, 'b': 3}, 'id': 'call_cpVH5hqk6t9G8GUumwLfCkMw', 'type': 'tool_call'}]
[2] multi({'a': 9, 'b': 3}) → 27.0
[{'name': 'devide', 'args': {'a': 27, 'b': 2}, 'id': 'call_ysIn7rUW9uVEetitmaJPDNEi', 'type': 'tool_call'}]
[3] devide({'a': 27, 'b': 2}) → 13.5
[{'name': 'minus', 'args': {'a': 13.5, 'b': 10}, 'id': 'call_COn6rGKBIosfSVeJCKVBuSjP', 'type': 'tool_call'}]
[4] minus({'a': 13.5, 'b': 10}) → 3.5
[]

최종 답변: 결과는 **3.5**입니다.


# RAG

In [69]:
import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message=".*langchain-community.*")   # sunset 안내 경고 숨김
os.environ.setdefault("USER_AGENT", "MyRAGBot/1.0")                    # WebBaseLoader 경고 방지

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
embedding_model_name = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

print("OpenAI 키:", "있음" if os.getenv("OPENAI_API_KEY") else "없음 (.env 확인)")
print("LLM 모델 :", default_model)
print("임베딩 모델:", embedding_model_name)

OpenAI 키: 있음
LLM 모델 : gpt-6-luna
임베딩 모델: text-embedding-3-small


In [70]:
data_txt = """프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.

독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.

일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해 있다.

영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가 유명하다.

이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과 트레비 분수가 유명하다.

미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로, 백악관과 링컨 기념관이 유명하다.

호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.

스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한 왕궁이 유명하다.
"""

with open("data/data.txt", "w", encoding="utf-8") as f:
    f.write(data_txt)
print("data.txt 생성:", len(data_txt), "글자")

data.txt 생성: 496 글자


In [ ]:
import chromadb

client = chromadb.PersistentClient(path='my_chroma_db1')
collection = client.get_or_create_collection(name="my_collection")

In [75]:
collection.add(
    documents=['사과', '바나나', '블루베리', '치킨', '배'],
    ids=['1', '2', '3', '4', '5'],
    metadatas=[{'type':'과일'},{'type':'과일'},{'type':'과일'},{'type':'음식'}, {'type':'바디'}]
)


C:\Users\Admin\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:23<00:00, 3.58MiB/s]


In [76]:
data = collection.get(ids='1')
data

{'ids': ['1'],
 'embeddings': None,
 'documents': ['사과'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'type': '과일'}]}

In [85]:
data = collection.upsert(
    ids = ['1'],
    documents=['초록사과'],
    metadatas={'type':'과일', 'color':'초록'}
)

In [80]:
collection.delete(ids=['1'])

{'deleted': 1}

In [86]:
collection.get()

{'ids': ['2', '3', '4', '5', '1'],
 'embeddings': None,
 'documents': ['바나나', '블루베리', '치킨', '배', '초록사과'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'type': '과일'},
  {'type': '과일'},
  {'type': '음식'},
  {'type': '바디'},
  {'color': '초록', 'type': '과일'}]}

In [124]:
client.delete_collection(name="my_book")

In [125]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

client = chromadb.PersistentClient(path='chroma_book_db')
openai_ef = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name=embedding_model_name,
)
books = client.get_or_create_collection(name="my_book", embedding_function=openai_ef)

In [126]:
books.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [127]:
books.add(
    ids = ['b1', 'b2', 'b3', 'b4'],
    documents = ['A wizard boy goes to a magic school', 'A detective solves a murder on a train',
                'Hobbits travel to destroy a powerful ring', 'A robot learns what it means to be human'],
    metadatas= [{'genre': 'fantasy', 'yaer': 1997}, {'genre': 'mystery', 'yaer': 1934}, {'genre': 'fantasy', 'yaer': 1954}, {'genre': 'sci-fi', 'yaer': 2005}]
)

In [130]:
books.upsert(
    ids=['b1', 'b2', 'b3', 'b4'],
    documents=[
        'A wizard boy goes to a magic school',
        'A detective solves a murder on a train',
        'Hobbits travel to destroy a powerful ring',
        'A robot learns what it means to be human'
    ],
    metadatas=[
        {'genre': 'fantasy', 'year': 1997},
        {'genre': 'mystery', 'year': 1934},
        {'genre': 'fantasy', 'year': 1954},
        {'genre': 'sci-fi', 'year': 2005}
    ]
)

In [131]:
books.get()

{'ids': ['b1', 'b2', 'b3', 'b4'],
 'embeddings': None,
 'documents': ['A wizard boy goes to a magic school',
  'A detective solves a murder on a train',
  'Hobbits travel to destroy a powerful ring',
  'A robot learns what it means to be human'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'genre': 'fantasy', 'year': 1997, 'yaer': 1997},
  {'genre': 'mystery', 'yaer': 1934, 'year': 1934},
  {'year': 1954, 'genre': 'fantasy', 'yaer': 1954},
  {'yaer': 2005, 'year': 2005, 'genre': 'sci-fi'}]}

In [132]:
books.get(where={'genre':'fantasy'}, include=['metadatas'])

{'ids': ['b1', 'b3'],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas'],
 'data': None,
 'metadatas': [{'genre': 'fantasy', 'year': 1997, 'yaer': 1997},
  {'yaer': 1954, 'genre': 'fantasy', 'year': 1954}]}

In [106]:
results = books.query(
    query_texts=["space and artificial intelligence"],      # 리스트 → 여러 질문 동시 입력 가능
    n_results=2,                # Top-k
    # where={"type": "fruit"},  # 메타데이터 조건으로 범위 제한
)

In [107]:
print(results)

{'ids': [['b4', 'b2']], 'embeddings': None, 'documents': [['A robot learns what it means to be human', 'A detective solves a murder on a train']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'genre': 'sci-fi', 'yaer': 2005}, {'genre': 'mystery', 'yaer': 1934}]], 'distances': [[0.587600588798523, 0.861982524394989]]}


In [108]:
books.update(
    ids=['b2'],
    metadatas={'year':1935}
)

In [109]:
books.get()

{'ids': ['b1', 'b2', 'b3', 'b4'],
 'embeddings': None,
 'documents': ['A wizard boy goes to a magic school',
  'A detective solves a murder on a train',
  'Hobbits travel to destroy a powerful ring',
  'A robot learns what it means to be human'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'genre': 'fantasy', 'yaer': 1997},
  {'genre': 'mystery', 'year': 1935, 'yaer': 1934},
  {'genre': 'fantasy', 'yaer': 1954},
  {'genre': 'sci-fi', 'yaer': 2005}]}

In [110]:
books.delete(ids=['b4'])

{'deleted': 1}

In [111]:
books.get()

{'ids': ['b1', 'b2', 'b3'],
 'embeddings': None,
 'documents': ['A wizard boy goes to a magic school',
  'A detective solves a murder on a train',
  'Hobbits travel to destroy a powerful ring'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'yaer': 1997, 'genre': 'fantasy'},
  {'yaer': 1934, 'year': 1935, 'genre': 'mystery'},
  {'genre': 'fantasy', 'yaer': 1954}]}

In [136]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/data.txt", encoding="utf-8")
documents = loader.load()

print("Document 수:", len(documents))
print("metadata   :", documents[0].metadata)
print("본문 앞부분:", documents[0].page_content[:60], "...")

Document 수: 1
metadata   : {'source': 'data/data.txt'}
본문 앞부분: 프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이  ...


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

if os.path.exists("data/data.pdf"):
    docs_pdf = PyPDFLoader("data/data.pdf").load()            # 페이지마다 Document 1개
    print("페이지 수:", len(docs_pdf), "| metadata:", docs_pdf[0].metadata)
else:
    print("data.pdf가 없습니다. 강의 자료실에서 내려받아 같은 폴더에 두세요.")

페이지 수: 1 | metadata: {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3189', 'creationdate': '2026-06-13T14:03:39+09:00', 'author': 'grovy', 'moddate': '2026-06-13T14:03:39+09:00', 'pdfversion': '1.4', 'source': 'data/data.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [134]:
from langchain_community.document_loaders import WebBaseLoader

docs_web = WebBaseLoader("https://ko.wikipedia.org/wiki/프랑스").load()
print(docs_web[0].metadata.get("title"), "|", len(docs_web[0].page_content), "글자")

프랑스 - 위키백과, 우리 모두의 백과사전 | 64597 글자


## Text splitter

In [140]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.split_documents(documents)
for i, chunk in enumerate(docs):
    print(f"Chunk {i+1}: {len(chunk.page_content)}글자 {chunk.page_content}")

Chunk 1: 65글자 프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.
Chunk 2: 50글자 독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.
Chunk 3: 53글자 일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해 있다.
Chunk 4: 53글자 영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가 유명하다.
Chunk 5: 63글자 이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과 트레비 분수가 유명하다.
Chunk 6: 65글자 미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로, 백악관과 링컨 기념관이 유명하다.
Chunk 7: 74글자 호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.
Chunk 8: 58글자 스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한 왕궁이 유명하다.


## embedding

In [142]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model=embedding_model_name)

q_vec = embeddings.embed_query('프랑스의 수도는 어디니?')
d_vecs = embeddings.embed_documents([chunk.page_content for chunk in docs])

print(len(q_vec))
print(len(d_vecs), len(d_vecs[0]))

1536
8 1536


In [147]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents= docs,
    embedding=embeddings,
    ids = [str(i+1) for i in range(len(docs))],
    collection_name='capital_collection',
    persist_directory='chroma_capital_db'
)